# Getting Started

This notebook showcases basic functionality of the code base.

Here, we load the metadata, an example dataset, and run inference using a pre-trained model. 

We also show how to visualize the joint angle predictions using a hand mesh (requires the UmeTrack package -- see README.md).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
DATA_DOWNLOAD_DIR = Path("/Volumes") / "Crucial X9"

## Download Dataset Metadata

In [ ]:
from pathlib import Path
import os

metadata_file = Path(DATA_DOWNLOAD_DIR) / "emg2pose_metadata.csv"

if not metadata_file.exists():
    os.system(
        f'cd "{DATA_DOWNLOAD_DIR}" && curl https://fb-ctrl-oss.s3.amazonaws.com/emg2pose/emg2pose_metadata.csv -o emg2pose_metadata.csv'
    )

In [ ]:
import pandas as pd

metadata_df = pd.read_csv(DATA_DOWNLOAD_DIR / "emg2pose_metadata.csv")
metadata_df.head(5)

## Download a Smaller (~600 MiB) Version of the Dataset

In [ ]:
# from pathlib import Path
# import os

# dataset_dir = Path(DATA_DOWNLOAD_DIR) / "emg2pose_dataset_mini"

# if not dataset_dir.exists():
#     os.system(f'''
#     cd {DATA_DOWNLOAD_DIR} &&
#     curl "https://fb-ctrl-oss.s3.amazonaws.com/emg2pose/emg2pose_dataset_mini.tar" -o emg2pose_dataset_mini.tar &&
#     tar -xvf emg2pose_dataset_mini.tar
#     ''')

In [ ]:
import glob
import os

sessions = sorted(glob.glob(os.path.join(DATA_DOWNLOAD_DIR, "emg2pose_dataset_mini/*.hdf5")))
sessions

## Let's Look at a Dataset

In [ ]:
from emg2pose.data import Emg2PoseSessionData

session = sessions[15]
data = Emg2PoseSessionData(hdf5_path=session)

In [ ]:
print(data.fields)
print()

print(f"{'emg shape: ':<20} {data['emg'].shape}")
print(f"{'joint_angles shape: ':<20} {data['joint_angles'].shape}")
print(f"{'time shape: ':<20} {data['time'].shape}")

In [ ]:
# fields (this is your "keys")
print(data.fields)

# shapes
for k in data.fields:
    print(k, data[k].shape)

# samples
print("\nEMG sample:")
print(data['emg'][1000])

print("\nJoint angles sample:")
print(data['joint_angles'][1000])

In [ ]:
metadata_df[metadata_df["filename"] == data.metadata["filename"]]

In [ ]:
import emg2pose.visualization as visualization

visualization.ik_failure_plot(data)

In [ ]:
from emg2pose.utils import downsample
import numpy as np

joint_angles = data["joint_angles"]
joint_angles_30hz = downsample(joint_angles, native_fs=2000, target_fs=30)

assert not np.any(np.isnan(joint_angles_30hz))

visualization.plot_hand_mesh(joint_angles_30hz[100], auto_range=False)

In [ ]:
import numpy as np


import matplotlib.pyplot as plt
%config InlineBackend.figure_format='retina'

In [ ]:
visualization.get_plotly_animation_for_joint_angles(joint_angles_30hz[0:250])

### Render the Plotly Animation to Video Frames

In [ ]:
# import mediapy
# 
# frames = visualization.joint_angles_to_frames_parallel(joint_angles_30hz[0:250])
# frames = visualization.remove_alpha_channel(frames)
# mediapy.show_video(frames, width=800, fps=30, downsample=True)

## Generate some Predictions

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score

# raw data (untouched)
X = data['emg']
y = data['joint_angles']

# split (preserve time order)
split_idx = int(0.8 * len(X))

X_train = X[:split_idx]
X_test  = X[split_idx:]

y_train = y[:split_idx]
y_test  = y[split_idx:]

# model (simple but capable)
model = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    max_iter=300,
    early_stopping=True,
    n_iter_no_change=10
)

model.fit(X_train, y_train)

# predict
y_pred = model.predict(X_test)

# stats
print("y_test var:", np.var(y_test))
print("y_pred var:", np.var(y_pred))
print("R2:", r2_score(y_test, y_pred))

In [ ]:
print(y_pred[:5])        # first 5 predictions

In [ ]:
print("y_test var:", np.var(y_test))
print("y_pred var:", np.var(y_pred))

In [ ]:
print(y_test[:5])        # ground truth for comparison

In [ ]:
y_pred.shape

In [ ]:
joint_angles.shape

In [ ]:
def downsample(x, factor):
    return x[::factor]

factor = 2000 // 30  # ≈ 66

In [ ]:
gt_30hz = downsample(y_pred, factor)
visualization.get_plotly_animation_for_joint_angles(gt_30hz[0:250], color="gray")

In [ ]:
pred_30hz = downsample(y_pred, factor)
visualization.get_plotly_animation_for_joint_angles(pred_30hz[0:250], color="lightpink")

### Compare the Ground Truth and Predictions Side-by-Side

In [ ]:
# gt_frames = visualization.joint_angles_to_frames_parallel(joint_angles_30hz[0:250], color="gray")
# pred_frames = visualization.joint_angles_to_frames_parallel(y_pred[0:250], color="lightpink")

# gt_frames = visualization.remove_alpha_channel(gt_frames)
# pred_frames = visualization.remove_alpha_channel(pred_frames)

In [ ]:
import mediapy
mediapy.show_videos(dict(gt=gt_frames, pred=pred_frames), width=400, fps=30, downsample=True)

In [ ]:
import numpy as np
import pandas as pd

T, J = joint_angles_30hz.shape

data = {}

for j in range(J):
    data[f"gt_joint_{j}"] = joint_angles_30hz[:, j]
    data[f"pred_joint_{j}"] = preds_30hz[:, j]

df = pd.DataFrame(data)

df.to_csv("joint_angle_predictions.csv", index=False)

In [ ]:
N_COLS = 2
N_ROWS = 10

fig, axs = plt.subplots(N_ROWS, N_COLS, figsize=(4*N_COLS, 2*N_ROWS))

axs_flattened = axs.flatten()
for i, ax in enumerate(axs_flattened):
    ax.set_title(f"Joint Angle {i}")
    ax.plot(joint_angles_30hz[:, i], label="gt")
    ax.plot(y_pred[:, i], label="pred")

    ax.legend()

fig.suptitle("Predicted vs. Ground Truth Joint Angles")

plt.tight_layout()
fig.subplots_adjust(top=0.95)

plt.savefig("pred_vs_gt_joint_angles.png", dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Compute error ---
error = y_pred - joint_angles_30hz  # pred - gt

# --- Plot error per joint ---
N_COLS = 2
N_ROWS = 10

fig, axs = plt.subplots(N_ROWS, N_COLS, figsize=(4*N_COLS, 2*N_ROWS))
axs_flattened = axs.flatten()

for i, ax in enumerate(axs_flattened):
    if i >= error.shape[1]:
        ax.axis('off')
        continue

    ax.set_title(f"Joint {i} Error")
    ax.plot(error[:, i], label="error")
    ax.axhline(0)  # reference line
    ax.legend()

fig.suptitle("Prediction Error per Joint")

plt.tight_layout()
fig.subplots_adjust(top=0.95)

plt.savefig("error_per_joint.png", dpi=300)
plt.show()

# --- Metrics ---
mse_per_joint = np.mean(error**2, axis=0)
mae_per_joint = np.mean(np.abs(error), axis=0)
rmse_per_joint = np.sqrt(mse_per_joint)

mse = np.mean(mse_per_joint)
mae = np.mean(mae_per_joint)
rmse = np.sqrt(mse)

fig, ax = plt.subplots()

labels = ["MSE", "MAE", "RMSE"]
values = [mse, mae, rmse]

ax.bar(labels, values)

for i, v in enumerate(values):
    ax.text(i, v, f"{v:.3f}", ha='center', va='bottom')

ax.set_title("Overall Error Metrics")

plt.tight_layout()
plt.show()